# JOINs en PySpark — Daniel Guzmán

**Semana:** 02  
**Actividad:** 02 — JOINs en PySpark  
**Dataset:** Financial Transactions  
**Objetivo:** Conectar las tablas del modelo financiero usando JOINs en PySpark y analizar fraude.

## Modelo de datos — Financial Transactions

Antes de hacer JOINs es importante entender cómo se relacionan las tablas del dataset.

| Tabla | Llave primaria | Llave foránea hacia |
|-------|----------------|---------------------|
| transactions_data | id | client_id → users_data.id, card_id → cards_data.id, mcc → mcc_codes.mcc, id → train_fraud_labels.id |
| users_data | id | — |
| cards_data | id | client_id → users_data.id |
| mcc_codes | mcc | — |
| train_fraud_labels | id | id → transactions_data.id |

### Diagrama de relaciones

transactions_data es la tabla principal.

users_data.id
→ transactions_data.client_id

cards_data.id
→ transactions_data.card_id

mcc_codes.mcc
→ transactions_data.mcc

train_fraud_labels.id
→ transactions_data.id

cards_data.client_id
→ users_data.id

In [0]:
MI_NOMBRE = "daniel"

VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

display(dbutils.fs.ls(VOL))

In [0]:
MI_NOMBRE = "daniel"

VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

display(dbutils.fs.ls(VOL))

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

# Tablas CSV
df_transactions = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/transactions_data.csv")
)

df_users = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/users_data.csv")
)

df_cards = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{VOL}/cards_data.csv")
)

# MCC codes JSON
df_mcc_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{VOL}/mcc_codes.json")
)

print("Tablas CSV y JSON MCC cargadas correctamente")

In [0]:
# Pivotar mcc_codes: de ancho a largo
mcc_cols = df_mcc_raw.columns

stack_expr = (
    f"stack({len(mcc_cols)}, "
    + ", ".join([f"'{c}', `{c}`" for c in mcc_cols])
    + ") as (mcc_str, description)"
)

df_mcc = (
    df_mcc_raw
    .select(F.expr(stack_expr))
    .withColumn("mcc", F.col("mcc_str").cast("int"))
    .drop("mcc_str")
)

print(f"Categorías MCC: {df_mcc.count():,}")
df_mcc.show(5, truncate=False)

In [0]:
try:
    df_fraud = spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")
    print("✓ Parquet cargado correctamente")
except Exception:
    print("⚠ Parquet no encontrado — convirtiendo desde JSON...")
    df_fraud = spark.read.option("multiLine", "true").json(f"{VOL}/train_fraud_labels.json")
    df_fraud.write.mode("overwrite").parquet(f"{VOL}/train_fraud_labels.parquet")
    df_fraud = spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")
    print("✓ Conversión completada y Parquet relanzado")

In [0]:
for nombre, dataframe in [
    ("transactions", df_transactions),
    ("users", df_users),
    ("cards", df_cards),
    ("mcc", df_mcc),
    ("fraud", df_fraud)
]:
    print(f"{nombre}: {dataframe.count():,} registros | {len(dataframe.columns)} columnas")

## Carga de tablas y formatos

En esta parte se cargaron las 5 tablas del modelo financiero:

- `transactions_data.csv`: tabla principal de transacciones.
- `users_data.csv`: información de clientes.
- `cards_data.csv`: información de tarjetas.
- `mcc_codes.json`: códigos de categoría de comercio.
- `train_fraud_labels.parquet`: etiquetas de fraude.

### ¿Por qué JSON se lee diferente al CSV?

CSV es un formato tabular: cada fila representa un registro y las columnas están separadas por delimitadores.

JSON puede tener estructuras más flexibles, anidadas o con múltiples niveles. En este caso, `mcc_codes.json` viene como un objeto donde cada clave es un código MCC y cada valor es la descripción. Por eso Spark lo leyó inicialmente como una fila con muchas columnas, y fue necesario convertirlo a formato largo con columnas `mcc` y `description`.

### ¿Qué hace multiLine?

La opción `multiLine=True` permite leer archivos JSON que están escritos en varias líneas. Sin esa opción, Spark puede interpretar cada línea como un JSON independiente y generar errores o una estructura incorrecta.

### ¿Qué ventajas tiene Parquet sobre JSON?

Parquet es un formato columnar, comprimido y optimizado para análisis. En Spark suele ser más eficiente que JSON porque lee solo las columnas necesarias, conserva tipos de datos y ocupa menos espacio. JSON es más flexible y legible, pero normalmente es menos eficiente para procesamiento analítico grande.

In [0]:
# Limpieza previa antes de los JOINs
df_transactions_clean = (
    df_transactions
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("transaction_date", F.to_timestamp(F.col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("client_id", "user_id")
    .withColumn("mcc", F.col("mcc").cast("int"))
)

# Verificar tipos de llaves
print("transactions - user_id:", df_transactions_clean.schema["user_id"].dataType)
print("users - id:", df_users.schema["id"].dataType)
print("transactions - card_id:", df_transactions_clean.schema["card_id"].dataType)
print("cards - id:", df_cards.schema["id"].dataType)
print("transactions - mcc:", df_transactions_clean.schema["mcc"].dataType)
print("mcc_codes - mcc:", df_mcc.schema["mcc"].dataType)

df_transactions_clean.select(
    "transaction_id",
    "user_id",
    "card_id",
    "mcc",
    "amount",
    "transaction_date"
).show(5, truncate=False)

## Limpieza previa antes del JOIN

Antes de hacer los JOINs se limpiaron y estandarizaron algunas columnas de `transactions_data`.

La columna `amount` fue convertida de string a double, eliminando símbolos como `$` y `,`.

La columna `date` se convirtió a timestamp y se creó como `transaction_date`.

También se renombró `id` a `transaction_id` y `client_id` a `user_id` para que la relación con `users_data` fuera más clara.

Al revisar los tipos de las llaves, se validó que `user_id`, `card_id` y `mcc` tuvieran tipos compatibles con las tablas relacionadas. Esto es importante porque si una llave queda como string y la otra como integer, el JOIN puede fallar o producir resultados incorrectos.

In [0]:
df_users_renamed = df_users.withColumnRenamed("id", "user_id")

df_tx_users = df_transactions_clean.join(
    df_users_renamed,
    on="user_id",
    how="inner"
)

tx_originales = df_transactions_clean.count()
tx_join_users = df_tx_users.count()
tx_perdidas_users = tx_originales - tx_join_users

print(f"Transactions originales: {tx_originales:,}")
print(f"Después del JOIN con users: {tx_join_users:,}")
print(f"¿Perdimos registros?: {tx_perdidas_users:,}")

df_tx_users.show(5, truncate=False)

## JOIN 1 — transactions + users

Se realizó un `INNER JOIN` entre `transactions_data` y `users_data` usando la llave `user_id`.

- Transacciones originales: **X**
- Transacciones después del JOIN con users: **Y**
- Registros perdidos: **Z**

Un `INNER JOIN` conserva únicamente los registros que tienen coincidencia en ambas tablas. Si una transacción tiene un `user_id` que no existe en `users_data`, esa transacción se pierde en el resultado.

Con este JOIN se agregaron columnas demográficas del cliente, como información de edad, género, ubicación u otros atributos disponibles en `users_data`.

In [0]:
df_cards_renamed = df_cards.withColumnRenamed("id", "card_id")

df_tx_users_cards = df_tx_users.join(
    df_cards_renamed,
    on="card_id",
    how="left"
)

sin_tarjeta = df_tx_users_cards.filter(F.col("card_type").isNull()).count()

print(f"Registros antes del JOIN con cards: {df_tx_users.count():,}")
print(f"Registros después del JOIN con cards: {df_tx_users_cards.count():,}")
print(f"Transacciones sin tarjeta asociada: {sin_tarjeta:,}")

df_tx_users_cards.show(5, truncate=False)

## JOIN 2 — users + cards

Se realizó un `LEFT JOIN` para agregar la información de tarjetas desde `cards_data`.

Usé `LEFT JOIN` porque quiero conservar todas las transacciones que ya venían del JOIN anterior, incluso si alguna no tiene una tarjeta asociada en `cards_data`.

Si usara `INNER JOIN`, se perderían las transacciones cuyo `card_id` no exista en la tabla de tarjetas. En análisis financiero esto puede ser riesgoso, porque una transacción sin tarjeta asociada también puede indicar un problema de calidad de datos o una relación incompleta.

Transacciones sin tarjeta asociada: 0

In [0]:
df_full = df_tx_users_cards.join(
    df_mcc,
    on="mcc",
    how="left"
)

sin_mcc = df_full.filter(F.col("description").isNull()).count()

print(f"Registros antes del JOIN con MCC: {df_tx_users_cards.count():,}")
print(f"Registros después del JOIN con MCC: {df_full.count():,}")
print(f"Transacciones sin categoría MCC asociada: {sin_mcc:,}")

df_full.groupBy("description") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(10) \
    .show(truncate=False)

## JOIN 3 — agregar categorías MCC

Se realizó un `LEFT JOIN` entre las transacciones enriquecidas y `mcc_codes`.

La columna `mcc` representa la categoría del comercio. Al unir con `mcc_codes`, se obtiene una descripción más entendible del tipo de comercio donde ocurrió la transacción.

Usé `LEFT JOIN` para conservar todas las transacciones, incluso si algún código MCC no tiene descripción asociada.

Transacciones sin categoría MCC asociada: 0

In [0]:
df_fraud_renamed = (
    df_fraud
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("target", "is_fraud")
)

df_final = df_full.join(
    df_fraud_renamed,
    on="transaction_id",
    how="left"
)

print("Distribución de labels:")
df_final.groupBy("is_fraud").count().show()

sin_label = df_final.filter(F.col("is_fraud").isNull()).count()

print(f"Registros antes del JOIN con fraud labels: {df_full.count():,}")
print(f"Registros después del JOIN con fraud labels: {df_final.count():,}")
print(f"Transacciones sin etiqueta de fraude: {sin_label:,}")

df_final.show(5, truncate=False)

## JOIN 4 — agregar etiquetas de fraude

Se realizó un `LEFT JOIN` entre el DataFrame enriquecido y `train_fraud_labels`.

La tabla de fraude permite identificar si una transacción fue marcada como fraudulenta o legítima. Se usó `LEFT JOIN` para conservar todas las transacciones, incluso si alguna no tiene etiqueta de fraude.

Esto es importante porque una transacción sin etiqueta también puede ser útil para análisis exploratorio, aunque no se pueda usar directamente como dato etiquetado para entrenamiento supervisado.

Transacciones sin etiqueta de fraude: 4,390,952

In [0]:
# Transacciones que NO tienen usuario registrado
tx_sin_usuario = df_transactions_clean.join(
    df_users_renamed,
    on="user_id",
    how="left_anti"
)

print(f"Transacciones sin usuario: {tx_sin_usuario.count():,}")

In [0]:
# Usuarios que NUNCA hicieron una transacción
usuarios_sin_tx = df_users_renamed.join(
    df_transactions_clean.select("user_id").distinct(),
    on="user_id",
    how="left_anti"
)

print(f"Usuarios sin transacciones: {usuarios_sin_tx.count():,}")

## Parte 5 — Anti JOIN

El `left_anti` permite encontrar registros de una tabla que no tienen coincidencia en otra.

Resultados:

- Transacciones sin usuario registrado: 0
- Usuarios sin transacciones: 781

Esto ayuda a evaluar la calidad del modelo de datos. Si existen transacciones sin usuario, puede indicar problemas de integridad referencial. Si existen usuarios sin transacciones, no necesariamente es un error; puede significar clientes registrados que aún no han realizado operaciones o registros demográficos sin actividad transaccional.

In [0]:
fraude_por_tipo_tarjeta = (
    df_final
    .groupBy("card_type")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == "Yes", 1).otherwise(0)).alias("total_fraudes")
    )
    .withColumn(
        "tasa_fraude",
        F.col("total_fraudes") / F.col("total_transacciones")
    )
    .orderBy(F.col("tasa_fraude").desc())
)

display(fraude_por_tipo_tarjeta)

## Pregunta 1 — Tipo de tarjeta con mayor tasa de fraude

Se calculó la tasa de fraude por tipo de tarjeta dividiendo el total de transacciones fraudulentas entre el total de transacciones de cada tipo de tarjeta.

El tipo de tarjeta con mayor tasa de fraude fue: 0.0015048

Este análisis permite identificar si ciertos tipos de tarjeta presentan mayor riesgo relativo de fraude.

In [0]:
fraude_por_mcc = (
    df_final
    .filter(F.col("is_fraud") == "Yes")
    .groupBy("mcc", "description")
    .agg(
        F.count("transaction_id").alias("total_fraudes")
    )
    .orderBy(F.col("total_fraudes").desc())
)

display(fraude_por_mcc.limit(10))

## Pregunta 2 — Categoría MCC con más fraude

Se filtraron las transacciones fraudulentas y luego se agruparon por código MCC y descripción de comercio.

La categoría MCC con mayor cantidad de transacciones fraudulentas fue: Department Stores, con MCC 5311 y 2,251 fraudes.

Este resultado ayuda a identificar categorías de comercio donde se concentra más fraude.

In [0]:
monto_fraude_vs_legitimo = (
    df_final
    .groupBy("is_fraud")
    .agg(
        F.count("transaction_id").alias("total"),
        F.avg("amount").alias("monto_promedio"),
        F.percentile_approx("amount", 0.5).alias("mediana_monto")
    )
    .orderBy("is_fraud")
)

display(monto_fraude_vs_legitimo)

## Pregunta 3 — Monto promedio fraudulento vs legítimo

Se comparó el monto promedio y la mediana de las transacciones según su etiqueta de fraude.

Este análisis permite revisar si las transacciones fraudulentas tienden a tener montos mayores, menores o similares frente a las transacciones legítimas.

Según el resultado, las transacciones fraudulentas tuvieron un monto promedio de 110.23, mientras que las legítimas tuvieron un monto promedio de 42.85.

In [0]:
df_final_enriched = (
    df_final
    .withColumn("year", F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
    .withColumn("day_of_week", F.dayofweek(F.col("transaction_date")))
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False)
    )
)

In [0]:
fraude_fin_semana = (
    df_final_enriched
    .groupBy("is_weekend")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == "Yes", 1).otherwise(0)).alias("total_fraudes")
    )
    .withColumn(
        "tasa_fraude",
        F.col("total_fraudes") / F.col("total_transacciones")
    )
    .orderBy(F.col("tasa_fraude").desc())
)

display(fraude_fin_semana)

## Pregunta 4 — Fraude en fin de semana vs entre semana

Se creó la columna `is_weekend` usando `dayofweek()`, donde domingo es `1` y sábado es `7`.

Luego se calculó la cantidad de fraudes y la tasa de fraude para fines de semana y días entre semana.

El resultado permite comparar si el fraude es relativamente más frecuente durante fines de semana o durante días laborales.

In [0]:
df_final_enriched = (
    df_final
    .withColumn("year", F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
    .withColumn("day_of_week", F.dayofweek(F.col("transaction_date")))
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False)
    )
)

In [0]:
fraude_fin_semana = (
    df_final_enriched
    .groupBy("is_weekend")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == "Yes", 1).otherwise(0)).alias("total_fraudes")
    )
    .withColumn(
        "tasa_fraude",
        F.col("total_fraudes") / F.col("total_transacciones")
    )
    .orderBy(F.col("tasa_fraude").desc())
)

display(fraude_fin_semana)

## Pregunta 4 — Fraude en fin de semana vs entre semana

Se creó la columna `is_weekend` usando `dayofweek()`, donde domingo es `1` y sábado es `7`.

Luego se calculó la cantidad de fraudes y la tasa de fraude para fines de semana y días entre semana.

El resultado muestra que la tasa de fraude fue ligeramente mayor en fin de semana.  
En fin de semana se observaron **4,080 fraudes** sobre **3,801,414 transacciones**, con una tasa aproximada de **0.001073**.

Entre semana se observaron **9,252 fraudes** sobre **9,504,501 transacciones**, con una tasa aproximada de **0.000973**.

Aunque entre semana hubo más fraudes en cantidad absoluta, el fin de semana tuvo mayor tasa relativa de fraude.

In [0]:
MI_NOMBRE = "daniel"

tabla_final = f"workspace.default.financial_final_{MI_NOMBRE}"

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabla_final)

print(f"✓ df_final guardado como {tabla_final}")
print(f"Filas: {df_final.count():,} | Columnas: {len(df_final.columns)}")
print(f"Columnas: {df_final.columns}")

## Persistencia en Delta

El DataFrame final `df_final` fue guardado como tabla Delta en Unity Catalog con el nombre:

`workspace.default.financial_final_daniel`

Se agregó el sufijo `daniel` para evitar pisar tablas de otros compañeros en el mismo catálogo.

La tabla final contiene **13,305,915 filas** y **40 columnas**. Esta tabla podrá reutilizarse en actividades posteriores sin repetir todos los JOINs.